<a href="https://colab.research.google.com/github/ziadkhalil04-jpg/ML-internship_test/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ziadkhalil04-jpg/ML-internship_test/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os
import json
import numpy as np
import pandas as pd

# Create outputs directory if it doesn't exist
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

# Generate baseline content audit action queue
np.random.seed(42)
n_pages = 100

urls = [f"https://example.com/blog/article-{i}" for i in range(1, n_pages + 1)]
archetypes = np.random.choice(['Informational', 'Transactional', 'Navigational', 'Comparison'], size=n_pages)
impressions_drop = np.random.uniform(0.05, 0.75, size=n_pages)
content_age_days = np.random.randint(90, 900, size=n_pages)
decay_score = (impressions_drop * 0.6) + ((content_age_days / 900) * 0.4)

df_queue = pd.DataFrame({
    'url': urls,
    'archetype': archetypes,
    'impressions_drop': np.round(impressions_drop, 3),
    'content_age_days': content_age_days,
    'decay_score': np.round(decay_score, 3)
})

# Assign Reason Codes and Action Recommendations
def assign_action(row):
    if row['decay_score'] >= 0.65:
        return 'REWRITE_HIGH_PRIORITY', 'DECAY_CRITICAL_SERP_DROP'
    elif row['decay_score'] >= 0.45:
        return 'REFRESH_METADATA_AND_LINKS', 'DECAY_MODERATE_STALE'
    elif row['content_age_days'] > 500 and row['impressions_drop'] < 0.2:
        return 'CONSOLIDATE_OR_PRUNE', 'LOW_ENGAGEMENT_HISTORICAL'
    else:
        return 'MAINTAIN_MONITOR', 'STABLE_PERFORMANCE'

df_queue[['recommended_action', 'reason_code']] = df_queue.apply(assign_action, axis=1, result_type='expand')
df_ranked = df_queue.sort_values(by='decay_score', ascending=False).reset_index(drop=True)

print("Top 5 Ranked Content Actions:")
print(df_ranked[['url', 'archetype', 'decay_score', 'recommended_action', 'reason_code']].head())

Top 5 Ranked Content Actions:
                                   url     archetype  decay_score  \
0  https://example.com/blog/article-91    Comparison        0.788   
1  https://example.com/blog/article-18    Comparison        0.747   
2  https://example.com/blog/article-24    Comparison        0.746   
3  https://example.com/blog/article-69    Comparison        0.737   
4   https://example.com/blog/article-4  Navigational        0.729   

      recommended_action               reason_code  
0  REWRITE_HIGH_PRIORITY  DECAY_CRITICAL_SERP_DROP  
1  REWRITE_HIGH_PRIORITY  DECAY_CRITICAL_SERP_DROP  
2  REWRITE_HIGH_PRIORITY  DECAY_CRITICAL_SERP_DROP  
3  REWRITE_HIGH_PRIORITY  DECAY_CRITICAL_SERP_DROP  
4  REWRITE_HIGH_PRIORITY  DECAY_CRITICAL_SERP_DROP  


## 2. Intended use and limits


### Intended Use
This action playbook is designed strictly as a **decision-support tool** for editorial and SEO teams. It prioritizes content audit queues based on measured decay patterns and algorithmic performance shifts.

### Operational Limits
* **Non-Production Execution**: Predictions and recommended actions must not trigger automated CMS edits, page deletions, or redirects without human validation.
* **Correlation vs. Causation**: High decay scores flag performance drops but do not isolate root causes (e.g., technical site outages vs. search algorithm updates).
* **Cost/Value Balance**: Low-value informational pages with high decay scores should be pruned or consolidated rather than allocated heavy engineering/writing resources.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Human Review Rules & The No-Go List

### Mandatory Human Review Triggers
1. Any page flagged for `CONSOLIDATE_OR_PRUNE` (deletion or URL redirect).
2. High-converting transactional pages generating core revenue.
3. Articles receiving significant external backlinks or active press coverage.

### The No-Go List (STRICTLY PROHIBITED FROM AUTOMATION)
* **Legal & Compliance Pages**: Privacy policies, terms of service, and regulatory disclosures.
* **Core Product & Conversion Pages**: Pricing tables, checkout funnels, and landing pages.
* **Brand/Navigational Hubs**: Homepage, company about page, and main service directories.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Monitoring & Retrain Triggers

The rules and models powering this playbook must undergo review and retraining upon encountering the following operational triggers:

1. **Major Search Engine Core Updates**: Any confirmed search algorithm update requires recalibrating impression baseline thresholds.
2. **False Positive Rate > 15%**: If editorial teams reject or override more than 15% of high-priority rewrite recommendations during weekly audits.
3. **Data Drift Threshold**: When overall distribution of `impressions_drop` metrics shifts by more than 20% compared to historical training data.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
# Export the ranked action queue CSV to work/outputs/
output_csv_path = '../outputs/ranked_action_queue.csv'
df_ranked.to_csv(output_csv_path, index=False)

# Export metrics summary JSON
summary_metrics = {
    'total_audited_pages': len(df_ranked),
    'critical_decay_pages': int((df_ranked['recommended_action'] == 'REWRITE_HIGH_PRIORITY').sum()),
    'refresh_recommended_pages': int((df_ranked['recommended_action'] == 'REFRESH_METADATA_AND_LINKS').sum()),
    'prune_recommended_pages': int((df_ranked['recommended_action'] == 'CONSOLIDATE_OR_PRUNE').sum()),
    'mean_decay_score': float(np.round(df_ranked['decay_score'].mean(), 3))
}

metrics_json_path = '../outputs/playbook_summary_metrics.json'
with open(metrics_json_path, 'w') as f:
    json.dump(summary_metrics, f, indent=2)

print(f"Action Queue exported successfully to: {output_csv_path}")
print(f"Summary Metrics exported successfully to: {metrics_json_path}")

Action Queue exported successfully to: ../outputs/ranked_action_queue.csv
Summary Metrics exported successfully to: ../outputs/playbook_summary_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.